In [1]:
!pip install transformers torch librosa opencv-python-headless --quiet

In [2]:
import torch
import torch.nn as nn
import librosa
import numpy as np
from transformers import BertTokenizer, BertModel
import matplotlib.pyplot as plt
import cv2

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------
# 1. MINI DATASET (3 samples)
# ----------------------------------------------------------

texts = [
    "I really loved this movie. It was amazing!",
    "The film was boring and too long.",
    "Nothing special… it was just okay."
]

labels = torch.tensor([2.5, -2.0, 0.0]).float().to(device)

# Fake audio files (sinus wave = simple demo)
sr = 16000
audios = [0.5*np.sin(np.linspace(0, 2000, sr)) for _ in range(3)]

# Fake images (colored squares)
images = [np.full((64,64,3), fill_value=i*80, dtype=np.uint8) for i in range(1,4)]


In [3]:
# ----------------------------------------------------------
# 2. TEXT ENCODER (BERT)
# ----------------------------------------------------------

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)

def encode_text(t):
    tokens = tokenizer(t, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        emb = bert(**tokens).last_hidden_state.mean(dim=1)
    return emb



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [4]:

# ----------------------------------------------------------
# 3. AUDIO FEATURES
# ----------------------------------------------------------

def encode_audio(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return torch.tensor(mfcc.mean(axis=1)).float().to(device)


In [5]:




# ----------------------------------------------------------
# 4. IMAGE ENCODER (Simple CNN)
# ----------------------------------------------------------

class ImgEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3,8,3,1,1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
    def forward(self, x):
        x = x/255.
        x = torch.tensor(x).permute(2,0,1).unsqueeze(0).float().to(device)
        return self.model(x).flatten()

img_encoder = ImgEncoder().to(device)

In [6]:



# ----------------------------------------------------------
# 5. FUSION MODEL
# ----------------------------------------------------------

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(768 + 20 + 8, 128),
            nn.ReLU(),
            nn.Linear(128,1)
        )
    def forward(self, t, a, v):
        x = torch.cat([t,a,v], dim=-1)
        return self.fc(x)

model = FusionNet().to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [8]:
!pip install transformers torch librosa opencv-python-headless --quiet

import torch
import torch.nn as nn
import librosa
import numpy as np
from transformers import BertTokenizer, BertModel
import cv2

device = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================================================
# 1. MINI DATASET (3 exemples simulés pour tester facilement)
# ===============================================================

texts = [
    "I really loved this movie. It was amazing!",
    "The film was boring and too long.",
    "Nothing special… it was just okay."
]

labels = torch.tensor([2.5, -2.0, 0.0]).float().to(device)

sr = 16000

# Fake audio wave (sinus simple)
audios = [0.5 * np.sin(np.linspace(0, 2000, sr)) for _ in range(3)]

# Fake images (petits carrés)
images = [np.full((64, 64, 3), fill_value=i * 80, dtype=np.uint8) for i in range(1, 4)]

# ===============================================================
# 2. TEXT ENCODING (BERT)
# ===============================================================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)

def encode_text(text):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        emb = bert(**tokens).last_hidden_state.mean(dim=1)
    return emb  # shape: [1, 768]


# ===============================================================
# 3. AUDIO ENCODING (MFCC)
# ===============================================================

def encode_audio(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    mfcc_mean = torch.tensor(mfcc.mean(axis=1)).float().to(device)  # shape: [20]
    return mfcc_mean


# ===============================================================
# 4. IMAGE ENCODER (Mini CNN)
# ===============================================================

class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 8, 3, 1, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)  # → [8]
        )

    def forward(self, img):
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).unsqueeze(0).float().to(device)
        return self.cnn(img).flatten()  # shape: [8]

img_encoder = ImageEncoder().to(device)


# ===============================================================
# 5. MULTIMODAL FUSION MODEL (fixé)
# ===============================================================

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(768 + 20 + 8, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, t, a, v):

        # ---- FIX: convert all to 2D tensors ----
        if t.dim() == 1:
            t = t.unsqueeze(0)
        if a.dim() == 1:
            a = a.unsqueeze(0)
        if v.dim() == 1:
            v = v.unsqueeze(0)

        x = torch.cat([t, a, v], dim=-1)  # now same dim ✔
        return self.fc(x)

model = FusionNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


# ===============================================================
# 6. TRAINING LOOP
# ===============================================================

print("Training...")
for epoch in range(80):
    total_loss = 0

    for i in range(3):
        t = encode_text(texts[i])
        a = encode_audio(audios[i])
        v = img_encoder(images[i])

        pred = model(t, a, v).squeeze()
        loss = loss_fn(pred, labels[i])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} - Loss: {total_loss:.4f}")

print("\nTraining completed!")


# ===============================================================
# 7. FINAL TEST PREDICTION
# ===============================================================

test_text = "This movie was absolutely fantastic!"
test_audio = audios[0]
test_image = images[1]

t = encode_text(test_text)
a = encode_audio(test_audio)
v = img_encoder(test_image)

pred = model(t, a, v).item()

print("\n--------------------------------")
print(" Predicted Sentiment Score =", pred)
print(" (range typically -3 = négatif → +3 = positif) ")
print("--------------------------------")


Training...
Epoch 0 - Loss: 65.6753
Epoch 10 - Loss: 2.4002
Epoch 20 - Loss: 0.0743
Epoch 30 - Loss: 0.0017
Epoch 40 - Loss: 0.0001
Epoch 50 - Loss: 0.0000
Epoch 60 - Loss: 0.0000
Epoch 70 - Loss: 0.0000

Training completed!

--------------------------------
 Predicted Sentiment Score = 1.4973965883255005
 (range typically -3 = négatif → +3 = positif) 
--------------------------------
